In [ ]:
import sys
sys.path.insert(0, "..")

import torch
import matplotlib.pyplot as plt
from config import Config
from src.tokenizer import Tokenizer
from src.dataset import download_wikitext2, make_dataloader, show_batch
from src.model import GPT
from src.generate import generate, load_from_checkpoint

In [ ]:
tok = Tokenizer()
paths = download_wikitext2("../data/wikitext2")
tokens = tok.encode_file(paths["train"])
loader = make_dataloader(tokens, context_length=32, batch_size=4, shuffle=True)
x, y = next(iter(loader))
print(f"Batch shape: x={x.shape}, y={y.shape}")
show_batch(x, y, tok)

In [ ]:
cfg = Config()
model = GPT(cfg)
print(f"Parameters: {model.num_params():,}")
for name, module in model.named_children():
    n = sum(p.numel() for p in module.parameters())
    print(f"  {name}: {n:,}")

In [ ]:
ckpt_path = "../checkpoints/step_001000.pt"   # update with your checkpoint
model = load_from_checkpoint(ckpt_path)
tok = Tokenizer()

for temp in [0.5, 1.0, 1.5]:
    print(f"\n--- temperature={temp} ---")
    print(generate(model, tok, prompt="The history of", max_new_tokens=80, temperature=temp))

In [ ]:
import torch.nn.functional as F
import math

def get_attention_map(model, tok, text):
    """Extract attention weights from the first head of the first block."""
    tokens = tok.encode(text)
    x = torch.tensor(tokens).unsqueeze(0)
    model.eval()

    with torch.no_grad():
        attn_weights = {}
        def hook(module, inp, out):
            B, T, C = inp[0].shape
            qkv = module.qkv(inp[0])
            q, k, v = qkv.split(C, dim=2)
            n_heads, d_head = module.n_heads, module.d_head
            q = q.view(B, T, n_heads, d_head).transpose(1, 2)
            k = k.view(B, T, n_heads, d_head).transpose(1, 2)
            scale = math.sqrt(d_head)
            w = (q @ k.transpose(-2, -1)) / scale
            mask = torch.triu(torch.ones(T, T), diagonal=1).bool()
            w = w.masked_fill(mask, float("-inf"))
            attn_weights["w"] = torch.softmax(w, dim=-1)[0, 0].detach()  # head 0

        handle = model.blocks[0].attn.register_forward_hook(hook)
        model(x)
        handle.remove()

    return attn_weights["w"], [tok.decode([t]) for t in tokens]

text = "The cat sat on the mat"
weights, labels = get_attention_map(model, tok, text)

plt.figure(figsize=(8, 6))
plt.imshow(weights.numpy(), cmap="Blues")
plt.xticks(range(len(labels)), labels, rotation=45, ha="right")
plt.yticks(range(len(labels)), labels)
plt.title("Attention weights — block 0, head 0")
plt.colorbar()
plt.tight_layout()
plt.show()